# OS detection: what survives when the model cannot see the host?

**Owner:** Chris. The OS-detection half of the netleak feature-leakage audit; the joint narrative is in [the report](../report/index.md), and the video half is in `02_video_services.ipynb`.

**Run:** *Restart kernel and run all*. Every cell reads from `results/` and `cache/`; nothing is refitted here, so a full run takes seconds. Regenerate the underlying experiments with `netleak grid -d os_detection` (~2.5 h).

Rung definitions come from `src/netleak/rungs.py`. This notebook never filters features itself.

In [1]:
import json
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from netleak import datasets, features, plots, results as results_io
from netleak.paths import default_paths
from netleak.rungs import RUNGS, rung_version

paths = default_paths()
spec = datasets.get('os_detection')
RUNG_ORDER = list(RUNGS)
MODEL_LABELS = {'logreg': 'Logistic regression', 'rf': 'Random forest', 'lgbm': 'LightGBM'}
SPLIT_LABELS = {'block': 'Contiguous block', 'random': 'Random'}
spec

DatasetSpec(name='os_detection', title='nPrint OS detection (13 classes)', benchmark_url='https://nprint.github.io/benchmarks/os_detection/nprint_os_detection.html', gdrive_id='1hlyGHqCgxPofS0HvCCl-ToKR-XbBtuPV', max_packets=100, n_classes=13, group_by='src_ip', leaderboard_bacc=0.771, label_from_metadata=<function _hard_label at 0x10ed4c400>, splits=('block', 'random'))

## 1. The dataset, and why it constrains the study

The benchmark defines a sample as 100 packets taken sequentially per source IP, labelled with the sending host's operating system. The label in the pcapML comment is `easylabel_hardlabel`; the benchmark task is the hard label.

In [2]:
fs = features.load(spec, paths.cache)
per_class = fs.meta.groupby('label').agg(samples=('sid', 'size'), hosts=('group', 'nunique'))
display(per_class.sort_values('samples', ascending=False))
display(Markdown(
    f"**{len(fs.meta):,} samples, {fs.meta['label'].nunique()} classes, "
    f"{fs.meta['group'].nunique()} source hosts.** Hosts appearing in more than one class: "
    f"**{(fs.meta.groupby('group')['label'].nunique() > 1).sum()}**."
))

,samples,hosts
label,,
mac-os-x,1000,1
ubuntu-14.4-32b,1000,1
ubuntu-14.4-64b,1000,1
ubuntu-16.4-32b,1000,1
ubuntu-16.4-64b,1000,1
ubuntu-server,1000,1
web-server,1000,1
windows-10,1000,1
windows-10-pro,1000,1


**12,439 samples, 13 classes, 13 source hosts.** Hosts appearing in more than one class: **0**.

**One host per class.** Thirteen classes, thirteen source addresses, and no host shared between classes. A host-grouped split would hold out whole classes, so `spec.splits` offers only `block` and `random`, and `splits.make_split` raises rather than scoring a class absent from training.

This is a limit on what the benchmark can answer, not a limit of the harness: on this dataset, *"knows Windows 10"* and *"knows the machine at that address"* cannot be separated. It also explains the R0 result below.

## 2. The restriction ladder and its cost in columns

In [3]:
rows = []
for name in RUNG_ORDER:
    rung = RUNGS[name]
    n = len(fs.matrix(name, np.arange(1))[1])
    rows.append({'Rung': name, 'Description': rung.description,
                 'Removed fields': ', '.join(sorted(rung.drop_fields)) or '—',
                 'Input columns': n})
display(pd.DataFrame(rows).set_index('Rung').style.format({'Input columns': '{:,}'}))

,Description,Removed fields,Input columns
Rung,,,
R0,Unrestricted: all nPrint IPv4+TCP bits,—,"56,290"
R1,Benchmark-legal: disallowed fields removed,"ipv4_dst, ipv4_src, tcp_ackn, tcp_dprt, tcp_seq, tcp_sprt","41,690"
R2,"Behaviour only: identifiers, checksums and TCP timestamps removed","ipv4_cksum, ipv4_dst, ipv4_id, ipv4_src, tcp_ackn, tcp_cksum, tcp_dprt, tcp_seq, tcp_sprt","36,890"
R3,Cheap baseline: six hand-picked header features,—,12


## 3. Results

Balanced accuracy on held-out test samples. Chance is 1/13 = 7.7%; the published best is 77.1% (AutoGluon, benchmark-legal features).

In [4]:
df = plots.primary_runs(results_io.load_all(paths.results))
os_df = df[df['dataset'] == 'os_detection']
table = os_df.pivot_table(index=['split', 'rung'], columns='model', values='balanced_accuracy')
table = (100 * table).reindex(
    pd.MultiIndex.from_product([list(SPLIT_LABELS), RUNG_ORDER], names=['split', 'rung'])
)
table = table[[m for m in MODEL_LABELS if m in table.columns]].rename(
    index=SPLIT_LABELS, columns=MODEL_LABELS)
display(table.style.format('{:.1f}', na_rep='—'))
display(Markdown(f'Cells available: **{len(os_df)}** of 24. Rung version '
                 f'`{rung_version()}`, seed 0.'))

Cells available: **7** of 24. Rung version `792434a48421`, seed 0.

In [5]:
fig, ax = plt.subplots(figsize=(7.5, 4), layout='constrained')
for model, label in MODEL_LABELS.items():
    for split, style in (('block', '-'), ('random', '--')):
        s = os_df[(os_df.model == model) & (os_df.split == split)].set_index('rung')
        s = 100 * s['balanced_accuracy'].reindex(RUNG_ORDER)
        if s.isna().all():
            continue
        ax.plot(RUNG_ORDER, s, style, marker='o', color=plots.MODEL_COLORS[model],
                label=f'{label} · {SPLIT_LABELS[split].lower()}')
ax.axhline(100 * spec.leaderboard_bacc, color='#52514e', lw=1)
ax.annotate(f'Published best {spec.leaderboard_bacc:.1%}', (0.02, 100 * spec.leaderboard_bacc + 1),
            fontsize=8, color='#52514e')
ax.axhline(100 / spec.n_classes, color='#898781', lw=1)
ax.annotate('Chance 7.7%', (0.02, 100 / spec.n_classes + 1), fontsize=8, color='#898781')
ax.set(ylim=(0, 104), ylabel='Balanced accuracy (%)', xlabel='Feature restriction rung',
       title='OS detection: accuracy against feature restriction')
ax.grid(axis='y', alpha=.2)
ax.legend(fontsize=8, loc='lower left')
plt.show()

/var/folders/w4/c7srzhyx44v_rr22dd7ncd480000gn/T/ipykernel_36115/2315932480.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Which header fields carry the surviving signal?

Permutation importance, grouped by nPrint field: every bit of a field is shuffled together and the drop in balanced accuracy is that field's score. Fields that duplicate each other mask one another, so a near-zero score means *redundant given the others*, not *uninformative*.

In [6]:
csvs = sorted((paths.results / 'os_detection' / 'importance').glob('*.csv'))
if not csvs:
    display(Markdown('_No importance run yet: `netleak importance -d os_detection --rung R1 --model lgbm --split block`._'))
else:
    imp = pd.read_csv(csvs[0])
    display(Markdown(f'From `{csvs[0].stem}`:'))
    display(imp.head(10).round(4))
    top = imp[imp.importance_mean.abs() >= 0.0005].head(12).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7.5, 0.35 * len(top) + 1.2), layout='constrained')
    ax.barh(top['field'], 100 * top['importance_mean'],
            xerr=100 * top['importance_std'], color=plots.MODEL_COLORS['logreg'], capsize=3)
    ax.set(xlabel='Drop in balanced accuracy (percentage points)',
           title='R1: fields carrying the surviving signal')
    ax.grid(axis='x', alpha=.2)
    plt.show()

_No importance run yet: `netleak importance -d os_detection --rung R1 --model lgbm --split block`._

## 5. What the representation costs

Feature count is a property of the representation; fit time is this implementation on one machine (M1 Pro, 10 cores). Neither is a universal speed claim.

In [7]:
cost = os_df.groupby(['rung', 'model']).agg(
    Features=('n_features', 'max'), Fit_seconds=('fit_seconds', 'median'),
    Extract_ms_per_sample=('extract_seconds_per_sample', 'median'),
).reset_index()
cost['Extract_ms_per_sample'] *= 1000
display(cost.pivot(index='rung', columns='model', values='Fit_seconds')
        .reindex(RUNG_ORDER).rename(columns=MODEL_LABELS).round(1)
        .rename_axis('Median fit time (s)'))
display(cost.groupby('rung')[['Features', 'Extract_ms_per_sample']].max()
        .reindex(RUNG_ORDER).round(2))

model,Logistic regression,Random forest
Median fit time (s),,
R0,43.8,4.4
R1,NaN,10.4
R2,NaN,NaN
R3,NaN,0.8


,Features,Extract_ms_per_sample
rung,,
R0,56290.0,0.83
R1,41690.0,0.83
R2,NaN,NaN
R3,12.0,0.72


## 6. Reading the results

*Filled in once the full grid has run; see the report's results page for the joint discussion.*

In [8]:
print('Rung version:', rung_version())
print('Encoder version:', json.loads((fs.root / 'manifest.json').read_text())['encoder_version'])
print('Cells:', len(os_df), 'of 24')

Rung version: 792434a48421
Encoder version: 1
Cells: 7 of 24
